In [1]:
import websocket
import json
import os
import threading
import time
from collections import namedtuple
from datetime import datetime
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SYMBOL = 'ETHUSDT'
WEBSOCKET_URL = f'wss://stream.binance.com:9443/ws/{SYMBOL.lower()}@depth10@100ms'
PARQUET_FILE = 'binance_orderbook.parquet'

OrderBook = namedtuple("OrderBook", ['ts', 'best_bid_px', 'best_bid_sz', 'best_ask_px', 'best_ask_sz'])

data_buffer = []
buffer_size = 100  

def write_to_parquet():
    if not data_buffer:
        return
    
    try:
        df = pd.DataFrame(data_buffer)
        
        data_buffer.clear()
        
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        schema = pa.schema([
            pa.field('timestamp', pa.timestamp('ns')),
            pa.field('bin_best_bid_px', pa.float64()),
            pa.field('bin_best_bid_sz', pa.float64()),
            pa.field('bin_best_ask_px', pa.float64()),
            pa.field('bin_best_ask_sz', pa.float64())
        ])
        
        table = pa.Table.from_pandas(df, schema=schema)
        
        if os.path.exists(PARQUET_FILE):
            existing = pq.read_table(PARQUET_FILE)
            combined = pa.concat_tables([existing, table])
            pq.write_table(combined, PARQUET_FILE)
        else:
            pq.write_table(table, PARQUET_FILE)
            
        print(f"Written {len(df)} records to {PARQUET_FILE}")
        
    except Exception as e:
        print(f"Error writing to parquet: {e}")
        data_buffer.extend([pd.DataFrame(data_buffer).to_dict('records')])

def get_bba(data):
    best_bid_px, best_bid_sz = data['bids'][0]
    best_bid_px, best_bid_sz = float(best_bid_px), float(best_bid_sz)

    best_ask_px, best_ask_sz = data['asks'][0]
    best_ask_px, best_ask_sz = float(best_ask_px), float(best_ask_sz)

    order_book = OrderBook(datetime.now(), best_bid_px, best_bid_sz, best_ask_px, best_ask_sz)
    return order_book

def on_message(ws, message):
    global last_write_time
    
    data = json.loads(message)
    order_book = get_bba(data)

    order_book_dict = {
        "timestamp": order_book.ts,
        "bin_best_bid_px": order_book.best_bid_px,
        "bin_best_bid_sz": order_book.best_bid_sz,
        "bin_best_ask_px": order_book.best_ask_px,
        "bin_best_ask_sz": order_book.best_ask_sz
    }
    
    data_buffer.append(order_book_dict)
    
    current_time = time.time()
    if len(data_buffer) >= buffer_size:
        write_to_parquet()
        last_write_time = current_time
    
    print(order_book)

def on_open(ws):
    print(f"WebSocket connected to {WEBSOCKET_URL}")
    print("Listening for order book updates...")

def on_close(ws, close_status_code, close_msg):
    print("Closing WebSocket connection...")
    if data_buffer:
        write_to_parquet()

def on_error(ws, error):
    print(f"WebSocket Error: {error}")

ws = websocket.WebSocketApp(
    WEBSOCKET_URL,
    on_message=on_message,
    on_open=on_open,
    on_close=on_close,
    on_error=on_error,
)

wst = threading.Thread(target=ws.run_forever)
wst.daemon = True
wst.start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nInterrupted by user")
    if data_buffer:
        write_to_parquet()
    ws.close()

WebSocket connected to wss://stream.binance.com:9443/ws/ethusdt@depth10@100ms
Listening for order book updates...
OrderBook(ts=datetime.datetime(2025, 12, 3, 17, 8, 38, 878018), best_bid_px=3049.2, best_bid_sz=9.6988, best_ask_px=3049.21, best_ask_sz=32.8603)
OrderBook(ts=datetime.datetime(2025, 12, 3, 17, 8, 38, 976413), best_bid_px=3049.2, best_bid_sz=9.6988, best_ask_px=3049.21, best_ask_sz=32.8603)
OrderBook(ts=datetime.datetime(2025, 12, 3, 17, 8, 39, 77438), best_bid_px=3049.2, best_bid_sz=5.0341, best_ask_px=3049.21, best_ask_sz=32.6504)
OrderBook(ts=datetime.datetime(2025, 12, 3, 17, 8, 39, 177787), best_bid_px=3049.2, best_bid_sz=4.9727, best_ask_px=3049.21, best_ask_sz=32.6504)
OrderBook(ts=datetime.datetime(2025, 12, 3, 17, 8, 39, 281434), best_bid_px=3049.2, best_bid_sz=4.9727, best_ask_px=3049.21, best_ask_sz=32.6504)
OrderBook(ts=datetime.datetime(2025, 12, 3, 17, 8, 39, 373309), best_bid_px=3049.2, best_bid_sz=4.9781, best_ask_px=3049.21, best_ask_sz=32.6504)
OrderBook(t